Imports and Setup

In [ ]:
# only run this cell if modules have not been installed yet for this environment
%pip install pandas numpy matplotlib seaborn scikit-learn

   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   --------------------------- ------------ 5.8/8.4 MB 30.5 MB/s eta 0:00:01
   ---------------------------------------- 8.4/8.4 MB 21.4 MB/s  0:00:00
   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
   -------- ------------------------------- 7.9/37.4 MB 36.2 MB/s eta 0:00:01
   ----------------- ---------------------- 16.5/37.4 MB 38.5 MB/s eta 0:00:01
   --------------------------- ------------ 25.4/37.4 MB 39.9 MB/s eta 0:00:01
   ------------------------------------ --- 34.3/37.4 MB 40.3 MB/s eta 0:00:01
   ---------------------------------------  37.2/37.4 MB 40.2 MB/s eta 0:00:01
   ---------------------------------------  37.2/37.4 MB 40.2 MB/s eta 0:00:01
   ---------------------------------------  37.2/37.4 MB 40.2 MB/s eta 0:00:01
   ---------------------------------------- 37.4/37.4 MB 24.7 MB/s  0:00:01

   ------ --------------------------------- 1/6 [scipy]
   ------ ---------------

In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    make_scorer, average_precision_score, roc_auc_score, f1_score,
    recall_score, precision_score, confusion_matrix, classification_report,
    precision_recall_curve, roc_curve, balanced_accuracy_score
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option('display.max_columns', 100)

Load the data

In [6]:
df = pd.read_csv('../data/diabetic_data.csv')
print(f"Data shape: {df.shape}")
df.head()

Data shape: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


Data Audit  - Verify the shape, types and missing data

In [7]:
print("Shape of the dataset:", df.shape)
print("\nColumns types:\n", df.dtypes.value_counts())

print("\nMissing values (NaN or '?'):")
missing_values = df.isin(['?']).sum() + df.isna().sum()
print(missing_values[missing_values > 0])
print("\nMissing values percentage:")
print((missing_values[missing_values > 0] / df.shape[0]) * 100)

Shape of the dataset: (101766, 50)

Columns types:
 str      37
int64    13
Name: count, dtype: int64

Missing values (NaN or '?'):
race                  2273
weight               98569
payer_code           40256
medical_specialty    49949
diag_1                  21
diag_2                 358
diag_3                1423
max_glu_serum        96420
A1Cresult            84748
dtype: int64

Missing values percentage:
race                  2.233555
weight               96.858479
payer_code           39.557416
medical_specialty    49.082208
diag_1                0.020636
diag_2                0.351787
diag_3                1.398306
max_glu_serum        94.746772
A1Cresult            83.277322
dtype: float64


Target distribution

In [8]:
print("Raw target value counts:\n", df['readmitted'].value_counts())
print("\nProportion of target values:\n", df['readmitted'].value_counts(normalize=True))

# Creates binary target (1 if readmitted within 30 days, 0 otherwise)
df["readmitted_in_30_days"] = (df["readmitted"] == "<30").astype(int)
print("\nBinary target value counts:\n", df['readmitted_in_30_days'].value_counts())

print("\nProportion of binary target values:\n", df['readmitted_in_30_days'].value_counts(normalize=True))

Raw target value counts:
 readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

Proportion of target values:
 readmitted
NO     0.539119
>30    0.349282
<30    0.111599
Name: proportion, dtype: float64

Binary target value counts:
 readmitted_in_30_days
0    90409
1    11357
Name: count, dtype: int64

Proportion of binary target values:
 readmitted_in_30_days
0    0.888401
1    0.111599
Name: proportion, dtype: float64


Dealing with duplicates and grouping

In [9]:
print("Exact duplicates in the dataset:", df.duplicated().sum())
print("Unique patients in the dataset:", df['patient_nbr'].nunique())
print("Total encounters in the dataset:", len(df))
print("Average encounters per patient:", len(df) / df['patient_nbr'].nunique())

# Patients with multiple encounters
multiple_encounters = df['patient_nbr'].value_counts()
print("\nPateints with 1 encounter:", (multiple_encounters == 1).sum())
print("Patients with multiple encounters:", (multiple_encounters > 1).sum())


Exact duplicates in the dataset: 0
Unique patients in the dataset: 71518
Total encounters in the dataset: 101766
Average encounters per patient: 1.4229424760200229

Pateints with 1 encounter: 54745
Patients with multiple encounters: 16773


Ethical exclusions

In [13]:
# Discharge codes for patients who died or were admitted to hospice
# 11 = expired, 13 = hospice/home, 14 = hospice/medical facility,
# 19, 20, 21 = expired at home/medical facility/unknown

death_hospice_discharge_codes = [11, 13, 14, 19, 20, 21]


print("\nPatients with discharge codes indicating death or hospice care:", df["discharge_disposition_id"].isin(death_hospice_discharge_codes).sum())

df = df[~df["discharge_disposition_id"].isin(death_hospice_discharge_codes)].copy()
print("\nPatients remaining after removing death/hospice discharge codes:", len(df))
print("Shape of the dataset after removing death/hospice discharge codes:", df.shape)



Patients with discharge codes indicating death or hospice care: 2423

Patients remaining after removing death/hospice discharge codes: 99343
Shape of the dataset after removing death/hospice discharge codes: (99343, 51)


Dropping columns

In [14]:
# Columns to drop based on data audit
columns_to_drop = [
    "encounter_id",  # Unique identifier for each encounter, not useful for modeling
    "patient_nbr",  # Unique identifier for each patient, not useful for modeling
    "weight",  # High percentage of missing values
    "payer_code",  # High percentage of missing values
    "medical_specialty",  # High percentage of missing values
    "examide",  # High percentage of missing values
    "citoglipton",  # High percentage of missing values
    "diag_1",  # High percentage of missing values
    "diag_2",  # High percentage of missing values
    "diag_3",  # High percentage of missing values
    "readmitted",  # Original target variable, replaced by binary target
]

# keeping patient_nbr separatley for grouping
patient_ids = df["patient_nbr"].copy()

# Dropping columns
df.drop(columns=columns_to_drop, inplace=True)
print("\nShape of the dataset after dropping columns:", df.shape)
print("\nColumns remaining after dropping:", df.columns.tolist())


Shape of the dataset after dropping columns: (99343, 40)

Columns remaining after dropping: ['race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted_in_30_days']


Handling missing values markes as '?'

In [17]:
# Replace '?' with NaN for easier handling of missing values
df.replace('?', np.nan, inplace=True)

# Check for remaining missing values
print("\nMissing values after dropping columns and replacing '?':")
print(df.isna().sum()[df.isna().sum() > 0])



Missing values after dropping columns and replacing '?':
race              2234
max_glu_serum    94191
A1Cresult        82509
dtype: int64


Converting age to numeric midpoint

In [41]:
# convert age bracket strings to ordinal numeric midpoints
age_mapping = {
    '[0-10)': 5,
    '[10-20)': 15,
    '[20-30)': 25,
    '[30-40)': 35,
    '[40-50)': 45,
    '[50-60)': 55,
    '[60-70)': 65,
    '[70-80)': 75,
    '[80-90)': 85,
    '[90-100)': 95,
}

print("Age values before mapping:", df['age'].unique())
df['age'] = df['age'].map(age_mapping)
print("Age values after mapping:", df['age'].unique())
print("Missing after mapping:", df['age'].isna().sum())

Age values before mapping: <StringArray>
[  '[0-10)',  '[10-20)',  '[20-30)',  '[30-40)',  '[40-50)',  '[50-60)',
  '[60-70)',  '[70-80)',  '[80-90)', '[90-100)']
Length: 10, dtype: str
Age values after mapping: [ 5 15 25 35 45 55 65 75 85 95]
Missing after mapping: 0


Feature types for pipeline

In [46]:
target = "readmitted_in_30_days"


x = df.drop(columns=[target])
y = df[target]

# numerical vs categorical columns
numerical_cols = x.select_dtypes(include=['number']).columns.tolist()
categorical_cols = x.select_dtypes(include=['object', 'string']).columns.tolist() 


print("Numerical columns:", len(numerical_cols), "\n", numerical_cols)
print("Categorical columns:", len(categorical_cols), "\n", categorical_cols)

Numerical columns: 12 
 ['age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']
Categorical columns: 27 
 ['race', 'gender', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']


Processing pipeline

In [47]:
# numeric - median and scale
numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

# categorical - msot frequent and one hot encode
categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore")),
])

# Combine pipelines into a column transformer
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numerical_cols),
    ("categorical", categorical_pipeline, categorical_cols),
])  

Cross validation 

In [48]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("Cross validation splitter with", cv.get_n_splits(), "splits")

Cross validation splitter with 5 splits


Scoring

In [49]:
scoring = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "f1_score": "f1",
    "recall": "recall",
    "precision": "precision",
    "balanced_accuracy": "balanced_accuracy",
}

Dummy baseline

In [50]:
dummy_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)),
])

dummy_cv_results = cross_validate(
    dummy_model, x, y, groups=patient_ids, cv=cv, scoring=scoring, return_train_score=True, n_jobs=-1
)

print("Dummy model: mean scores across 5 folds:")
for metric, scores in dummy_cv_results.items():
    if metric.startswith("test_"):
        print(f"{metric[5:]}: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

Dummy model: mean scores across 5 folds:
pr_auc: 0.1139 ± 0.0000
roc_auc: 0.5000 ± 0.0000
f1_score: 0.0000 ± 0.0000
recall: 0.0000 ± 0.0000
precision: 0.0000 ± 0.0000
balanced_accuracy: 0.5000 ± 0.0000


Logistic Regression baseline

In [51]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE)),
])

logistic_cv_results = cross_validate(
    logistic_model, x, y, groups=patient_ids, cv=cv, scoring=scoring, return_train_score=True, n_jobs=-1
)

print("Logistic Regression model: mean scores across 5 folds:")
for metric, scores in logistic_cv_results.items():
    if metric.startswith("test_"):
        print(f"{metric[5:]}: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

Logistic Regression model: mean scores across 5 folds:
pr_auc: 0.2027 ± 0.0062
roc_auc: 0.6467 ± 0.0022
f1_score: 0.2618 ± 0.0024
recall: 0.5186 ± 0.0061
precision: 0.1751 ± 0.0016
balanced_accuracy: 0.6023 ± 0.0028


Results

In [53]:
results_table = pd.DataFrame({
    "Dummy": {m: f"{dummy_cv_results[f'test_{m}'].mean():.4f} ± {dummy_cv_results[f'test_{m}'].std():.4f}" for m in scoring},
    "LogReg (L2)": {m: f"{logistic_cv_results[f'test_{m}'].mean():.4f} ± {logistic_cv_results[f'test_{m}'].std():.4f}" for m in scoring},
})

print(results_table)

                             Dummy      LogReg (L2)
pr_auc             0.1139 ± 0.0000  0.2027 ± 0.0062
roc_auc            0.5000 ± 0.0000  0.6467 ± 0.0022
f1_score           0.0000 ± 0.0000  0.2618 ± 0.0024
recall             0.0000 ± 0.0000  0.5186 ± 0.0061
precision          0.0000 ± 0.0000  0.1751 ± 0.0016
balanced_accuracy  0.5000 ± 0.0000  0.6023 ± 0.0028
